<a href="https://colab.research.google.com/github/shubhishukla8176-arch/exfoliation-energy-ml/blob/main/notebooks/05_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os
import numpy as np
import pandas as pd

from google.colab import drive

drive.mount('/content/drive')

# Main project folder
project_folder = "/content/drive/MyDrive/MASC520_Project"

# Load composition descriptors
composition_path = os.path.join(
    project_folder,
    "composition_features",
    "composition_descriptors.csv"
)

composition_df = pd.read_csv(composition_path)

# Load structural descriptors
structure_path = os.path.join(
    project_folder,
    "structure_features",
    "structure_descriptors.csv"
)

structure_df = pd.read_csv(structure_path)

print("Both datasets loaded successfully!")

print("\nComposition dataset shape:", composition_df.shape)
print("Structure dataset shape:", structure_df.shape)

print("\nComposition columns:")
print(composition_df.columns.tolist())

print("\nStructure columns:")
print(structure_df.columns.tolist())


Mounted at /content/drive
Both datasets loaded successfully!

Composition dataset shape: (813, 218)
Structure dataset shape: (813, 24)

Composition columns:
['jid', 'fraction_Ag', 'fraction_Al', 'fraction_As', 'fraction_Au', 'fraction_B', 'fraction_Ba', 'fraction_Be', 'fraction_Bi', 'fraction_Br', 'fraction_C', 'fraction_Ca', 'fraction_Cd', 'fraction_Ce', 'fraction_Cl', 'fraction_Co', 'fraction_Cr', 'fraction_Cu', 'fraction_Dy', 'fraction_Er', 'fraction_F', 'fraction_Fe', 'fraction_Ga', 'fraction_Ge', 'fraction_H', 'fraction_Hf', 'fraction_Hg', 'fraction_Ho', 'fraction_I', 'fraction_In', 'fraction_Ir', 'fraction_K', 'fraction_Kr', 'fraction_Li', 'fraction_Lu', 'fraction_Mg', 'fraction_Mn', 'fraction_Mo', 'fraction_N', 'fraction_Na', 'fraction_Nb', 'fraction_Nd', 'fraction_Ni', 'fraction_Np', 'fraction_O', 'fraction_Os', 'fraction_P', 'fraction_Pb', 'fraction_Pd', 'fraction_Pr', 'fraction_Pt', 'fraction_Pu', 'fraction_Rb', 'fraction_Re', 'fraction_Rh', 'fraction_Ru', 'fraction_S', 'frac

In [2]:

print("Composition duplicate IDs:",
      composition_df["jid"].duplicated().sum())

print("Structure duplicate IDs:",
      structure_df["jid"].duplicated().sum())

print("\nMaterials in composition dataset:",
      composition_df["jid"].nunique())

print("Materials in structure dataset:",
      structure_df["jid"].nunique())

# Find IDs that are missing from either dataset
composition_ids = set(composition_df["jid"])
structure_ids = set(structure_df["jid"])

print("\nOnly in composition dataset:",
      len(composition_ids - structure_ids))

print("Only in structure dataset:",
      len(structure_ids - composition_ids))

print("Present in both datasets:",
      len(composition_ids & structure_ids))


Composition duplicate IDs: 0
Structure duplicate IDs: 0

Materials in composition dataset: 813
Materials in structure dataset: 813

Only in composition dataset: 0
Only in structure dataset: 0
Present in both datasets: 813


In [4]:

# Identify common material IDs
common_ids = sorted(composition_ids & structure_ids)

# Keep only common materials
composition_common = composition_df[
    composition_df["jid"].isin(common_ids)
].copy()

structure_common = structure_df[
    structure_df["jid"].isin(common_ids)
].copy()

# Sort both datasets in the same order
composition_common = composition_common.sort_values(
    "jid"
).reset_index(drop=True)

structure_common = structure_common.sort_values(
    "jid"
).reset_index(drop=True)

print("Common materials:", len(common_ids))

print(
    "Composition IDs match structure IDs:",
    composition_common["jid"].equals(structure_common["jid"])
)


Common materials: 813
Composition IDs match structure IDs: True


In [5]:

# Compare target values for matching material IDs

target_comparison = composition_common[
    ["jid", "exfoliation_energy"]
].merge(
    structure_common[["jid", "exfoliation_energy"]],
    on="jid",
    suffixes=("_composition", "_structure"),
    validate="one_to_one"
)

target_difference = np.abs(
    target_comparison["exfoliation_energy_composition"]
    - target_comparison["exfoliation_energy_structure"]
)

print("Maximum target difference:", target_difference.max())

print(
    "Are all target values identical?",
    np.allclose(
        target_comparison["exfoliation_energy_composition"],
        target_comparison["exfoliation_energy_structure"]
    )
)


Maximum target difference: 0.0
Are all target values identical? True


In [6]:

# Remove target from structure table before merging
structure_input = structure_common.drop(
    columns=["exfoliation_energy"]
)

# Combine composition and structure features
combined_df = composition_common.merge(
    structure_input,
    on="jid",
    how="inner",
    validate="one_to_one"
)

print("Combined dataset created!")

print("Number of materials:", len(combined_df))
print("Number of columns:", len(combined_df.columns))

display(combined_df.head())


Combined dataset created!
Number of materials: 813
Number of columns: 240


,jid,fraction_Ag,fraction_Al,fraction_As,fraction_Au,fraction_B,fraction_Ba,fraction_Be,fraction_Bi,fraction_Br,...,spacegroup_symbol,crystal_system,coordination_mean,coordination_min,coordination_max,coordination_std,weighted_coordination_mean,nearest_neighbor_distance_mean,nearest_neighbor_distance_min,nearest_neighbor_distance_max
0,JVASP-10,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,P-3m1,trigonal,4.000000,3,6,1.414214,4.000000,2.492265,2.492265,2.492265
1,JVASP-10173,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,P-3m1,trigonal,4.500000,3,6,1.500000,4.500000,3.055624,2.958314,3.202764
2,JVASP-10177,0.0,0.0,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,...,P2_1/m,monoclinic,2.666667,2,4,0.942809,2.666667,1.772676,1.702995,2.042051
3,JVASP-10209,0.0,0.0,0.400000,0.0,0.0,0.0,0.0,0.0,0.0,...,Cmcm,orthorhombic,8.800000,6,12,1.939072,8.800000,2.841992,2.455154,3.459948
4,JVASP-10571,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,P-1,triclinic,5.428571,3,10,2.060315,5.428571,2.717144,2.438011,3.441191


In [7]:

target_column = "exfoliation_energy"

# Composition-only dataset
composition_model_df = composition_common.copy()

# Structure-only dataset
structure_model_df = structure_common.copy()

# Combined dataset
combined_model_df = combined_df.copy()

# Check all three datasets
print("Composition-only:", composition_model_df.shape)
print("Structure-only:", structure_model_df.shape)
print("Combined:", combined_model_df.shape)


Composition-only: (813, 218)
Structure-only: (813, 24)
Combined: (813, 240)


In [8]:

datasets = {
    "Composition-only": composition_model_df,
    "Structure-only": structure_model_df,
    "Combined": combined_model_df
}

for name, dataset in datasets.items():

    print("\n", "=" * 50)
    print(name)

    print("Rows:", len(dataset))
    print("Missing values:", dataset.isnull().sum().sum())

    print("\nColumns with missing values:")
    print(dataset.columns[
        dataset.isnull().any()
    ].tolist())



Composition-only
Rows: 813
Missing values: 0

Columns with missing values:
[]

Structure-only
Rows: 813
Missing values: 0

Columns with missing values:
[]

Combined
Rows: 813
Missing values: 0

Columns with missing values:
[]


In [9]:

# Create output folder
model_folder = os.path.join(
    project_folder,
    "model_ready_data"
)

os.makedirs(model_folder, exist_ok=True)

# Save datasets
composition_model_df.to_csv(
    os.path.join(model_folder, "composition_only.csv"),
    index=False
)

structure_model_df.to_csv(
    os.path.join(model_folder, "structure_only.csv"),
    index=False
)

combined_model_df.to_csv(
    os.path.join(model_folder, "combined_features.csv"),
    index=False
)

print("All three model datasets saved successfully!")
print("Saved location:", model_folder)


All three model datasets saved successfully!
Saved location: /content/drive/MyDrive/MASC520_Project/model_ready_data
